In [2]:
import os
import time
import math
import statistics
import traceback
import arcpy
import numpy as np
from arcpy.sa import *
from tqdm import tqdm

# -------------------------------------------------------------------
# WHAT THIS SCRIPT DOES (4 lines)
# -------------------------------------------------------------------
# Fast crown-parameter sweep tuned to KEEP BIG CROWNS while still producing MANY crowns.
# Workflow unchanged: canopy mask → smoothing → local-max surface → seeds → watershed → polygons → optional PAEK smoothing.
# Scores runs using Perimeter (Shape_Length) + Area (Shape_Area) thresholds (P95 + Max), then picks BEST = max n among PASS.
# Writes an HTML report + a GDB results table; uses a robust seed-density gate (coarse resample + numpy) to avoid COUNT/SUM.
# -------------------------------------------------------------------

# -----------------------------------------------------------------------------
# ENVIRONMENT
# -----------------------------------------------------------------------------
arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")
arcpy.CheckOutExtension("3D")
arcpy.env.parallelProcessingFactor = "75%"

def stamp(msg: str):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

def exists_or_fail(path: str, what: str):
    if not arcpy.Exists(path):
        raise RuntimeError(f"{what} missing: {path}")

def safe_name(s: str) -> str:
    out = "".join(ch if ch.isalnum() else "_" for ch in s)
    return out[:60]

def raster_minmax(ras_path: str):
    mn = arcpy.management.GetRasterProperties(ras_path, "MINIMUM")[0]
    mx = arcpy.management.GetRasterProperties(ras_path, "MAXIMUM")[0]
    try:
        return float(mn), float(mx)
    except:
        return mn, mx

def percentile(sorted_vals, p: float):
    n = len(sorted_vals)
    if n == 0:
        return 0.0
    if n == 1:
        return float(sorted_vals[0])
    r = (p / 100.0) * (n - 1)
    lo = int(math.floor(r))
    hi = int(math.ceil(r))
    if lo == hi:
        return float(sorted_vals[lo])
    w = r - lo
    return float(sorted_vals[lo] * (1 - w) + sorted_vals[hi] * w)

def summarize(values):
    if not values:
        return {
            "n": 0, "min": 0, "max": 0, "mean": 0, "std": 0,
            "p01": 0, "p05": 0, "p25": 0, "p50": 0, "p75": 0, "p95": 0, "p99": 0
        }
    vals = sorted(values)
    n = len(vals)
    mean = sum(vals) / n
    std = statistics.pstdev(vals) if n > 1 else 0.0
    return {
        "n": n,
        "min": float(vals[0]),
        "max": float(vals[-1]),
        "mean": float(mean),
        "std": float(std),
        "p01": percentile(vals, 1),
        "p05": percentile(vals, 5),
        "p25": percentile(vals, 25),
        "p50": percentile(vals, 50),
        "p75": percentile(vals, 75),
        "p95": percentile(vals, 95),
        "p99": percentile(vals, 99),
    }

def ensure_results_table(gdb: str, table_name: str):
    tpath = os.path.join(gdb, table_name)
    if arcpy.Exists(tpath):
        return tpath
    arcpy.management.CreateTable(gdb, table_name)

    fields = [
        ("CELL","DOUBLE",None),
        ("HMIN","DOUBLE",None),
        ("SMOOTH","LONG",None),
        ("FMAX","LONG",None),
        ("EPS","DOUBLE",None),

        ("SEED_RATIO","DOUBLE",None),

        ("CROWN_N","LONG",None),

        ("MEAN_PERIM_M","DOUBLE",None),
        ("P95_PERIM_M","DOUBLE",None),
        ("MAX_PERIM_M","DOUBLE",None),

        ("MEAN_AREA_M2","DOUBLE",None),
        ("P95_AREA_M2","DOUBLE",None),
        ("MAX_AREA_M2","DOUBLE",None),

        ("PASS","TEXT",10),
        ("SECONDS","DOUBLE",None),
        ("OUT_FC","TEXT",255),
        ("STATUS","TEXT",40),
        ("ERRMSG","TEXT",255),
    ]

    for fn, ft, fl in fields:
        if ft == "TEXT":
            arcpy.management.AddField(tpath, fn, ft, field_length=fl)
        else:
            arcpy.management.AddField(tpath, fn, ft)

    return tpath

def log_result(table_path, row):
    fields = [
        "CELL","HMIN","SMOOTH","FMAX","EPS",
        "SEED_RATIO","CROWN_N",
        "MEAN_PERIM_M","P95_PERIM_M","MAX_PERIM_M",
        "MEAN_AREA_M2","P95_AREA_M2","MAX_AREA_M2",
        "PASS","SECONDS","OUT_FC","STATUS","ERRMSG"
    ]
    with arcpy.da.InsertCursor(table_path, fields) as ic:
        ic.insertRow(row)

def collect_perim_and_area(fc: str, sample_max_features: int = 0):
    """
    Collects Shape_Length (m) and Shape_Area (m²).
    For stable P95/max, keep sample_max_features=0 unless feature count is enormous.
    """
    perims, areas = [], []
    n = int(arcpy.management.GetCount(fc)[0])
    if n == 0:
        return perims, areas

    step = 1
    if sample_max_features and n > sample_max_features:
        step = max(1, n // sample_max_features)

    i = 0
    with arcpy.da.SearchCursor(fc, ["SHAPE@LENGTH", "SHAPE@AREA"]) as cur:
        for (L, A) in cur:
            if step > 1 and (i % step) != 0:
                i += 1
                continue
            perims.append(float(L))
            areas.append(float(A))
            i += 1
    return perims, areas

def write_html_report(html_path, meta, run_rows, best_row=None, pass_row=None):
    def esc(s):
        s = "" if s is None else str(s)
        return (s.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
                 .replace('"',"&quot;").replace("'","&#39;"))
    def fmt(x, nd=2):
        try:
            return f"{float(x):.{nd}f}"
        except:
            return esc(x)

    lines = []
    lines.append("<!DOCTYPE html><html><head><meta charset='utf-8'>")
    lines.append("<title>Crown sweep report (big crowns + many crowns)</title>")
    lines.append("""
<style>
body{font-family:Arial,Helvetica,sans-serif;margin:20px;line-height:1.35}
.small{color:#555;font-size:0.95em}
table{border-collapse:collapse;width:100%;margin:12px 0}
th,td{border:1px solid #ddd;padding:6px 8px;font-size:0.92em;vertical-align:top}
th{background:#f6f6f6;text-align:left}
.good{background:#ecffef}
.bad{background:#ffecec}
.mono{font-family:ui-monospace, SFMono-Regular, Menlo, Consolas, monospace}
</style>
""")
    lines.append("</head><body>")
    lines.append("<h1>Crown parameter sweep report</h1>")
    lines.append("<h3>Goal: big crowns + still many crowns (PASS uses P95+Max; BEST = max n among PASS)</h3>")
    lines.append(f"<div class='small'>Generated: {esc(meta['generated_at'])}</div>")

    lines.append("<h2>Inputs & thresholds</h2><table>")
    for k in meta.keys():
        lines.append(f"<tr><th>{esc(k)}</th><td class='mono'>{esc(meta.get(k))}</td></tr>")
    lines.append("</table>")

    lines.append("<h2>Best run (overall)</h2>")
    if best_row:
        lines.append("<table>")
        lines.append(f"<tr><th>Params</th><td class='mono'>CELL={best_row['CELL']}, HMIN={best_row['HMIN']}, SMOOTH={best_row['SMOOTH']}, FMAX={best_row['FMAX']}, EPS={best_row['EPS']}</td></tr>")
        lines.append(f"<tr><th>Status</th><td>{esc(best_row['status'])}</td></tr>")
        lines.append(f"<tr><th>PASS?</th><td>{esc(best_row['pass_flag'])}</td></tr>")
        lines.append(f"<tr><th>n</th><td>{best_row['n']:,}</td></tr>")
        lines.append(f"<tr><th>P95 / Max Perimeter (m)</th><td>{fmt(best_row['perim']['p95'])} / {fmt(best_row['perim']['max'])}</td></tr>")
        lines.append(f"<tr><th>P95 / Max Area (m²)</th><td>{fmt(best_row['area']['p95'])} / {fmt(best_row['area']['max'])}</td></tr>")
        lines.append(f"<tr><th>Output FC</th><td class='mono'>{esc(best_row['out_fc'])}</td></tr>")
        lines.append("</table>")
    else:
        lines.append("<p>No successful run produced crowns.</p>")

    if pass_row:
        lines.append("<h2>Best PASS run (max n among PASS)</h2>")
        lines.append("<table>")
        lines.append(f"<tr><th>Params</th><td class='mono'>CELL={pass_row['CELL']}, HMIN={pass_row['HMIN']}, SMOOTH={pass_row['SMOOTH']}, FMAX={pass_row['FMAX']}, EPS={pass_row['EPS']}</td></tr>")
        lines.append(f"<tr><th>n</th><td>{pass_row['n']:,}</td></tr>")
        lines.append(f"<tr><th>P95 / Max Perimeter (m)</th><td>{fmt(pass_row['perim']['p95'])} / {fmt(pass_row['perim']['max'])}</td></tr>")
        lines.append(f"<tr><th>P95 / Max Area (m²)</th><td>{fmt(pass_row['area']['p95'])} / {fmt(pass_row['area']['max'])}</td></tr>")
        lines.append(f"<tr><th>Output FC</th><td class='mono'>{esc(pass_row['out_fc'])}</td></tr>")
        lines.append("</table>")

    lines.append("<h2>All runs</h2>")
    lines.append("<div class='small'>Green rows PASS. BEST(PASS) = maximum n. ERRMSG truncated.</div>")
    lines.append("<table>")
    headers = [
        "CELL","HMIN","SMOOTH","FMAX","EPS","status","seconds","seed_ratio",
        "PASS","n",
        "perim_mean","perim_p95","perim_max",
        "area_mean","area_p95","area_max",
        "out_fc","errmsg"
    ]
    lines.append("<tr>" + "".join(f"<th>{h}</th>" for h in headers) + "</tr>")
    for r in run_rows:
        cls = "good" if r.get("pass_flag") == "PASS" else ("bad" if r.get("status","").startswith("ERROR") else "")
        lines.append(f"<tr class='{cls}'>" + "".join([
            f"<td>{esc(r.get('CELL'))}</td>",
            f"<td>{esc(r.get('HMIN'))}</td>",
            f"<td>{esc(r.get('SMOOTH'))}</td>",
            f"<td>{esc(r.get('FMAX'))}</td>",
            f"<td>{esc(r.get('EPS'))}</td>",
            f"<td>{esc(r.get('status'))}</td>",
            f"<td>{fmt(r.get('seconds',0),1)}</td>",
            f"<td>{fmt(r.get('seed_ratio',0),6)}</td>",
            f"<td>{esc(r.get('pass_flag',''))}</td>",
            f"<td>{int(r.get('n',0)):,}</td>",
            f"<td>{fmt(r.get('perim',{}).get('mean',0))}</td>",
            f"<td>{fmt(r.get('perim',{}).get('p95',0))}</td>",
            f"<td>{fmt(r.get('perim',{}).get('max',0))}</td>",
            f"<td>{fmt(r.get('area',{}).get('mean',0))}</td>",
            f"<td>{fmt(r.get('area',{}).get('p95',0))}</td>",
            f"<td>{fmt(r.get('area',{}).get('max',0))}</td>",
            f"<td class='mono'>{esc(r.get('out_fc',''))}</td>",
            f"<td class='mono'>{esc(r.get('errmsg',''))}</td>",
        ]) + "</tr>")
    lines.append("</table></body></html>")

    os.makedirs(os.path.dirname(html_path), exist_ok=True)
    with open(html_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

# -----------------------------------------------------------------------------
# INPUTS
# -----------------------------------------------------------------------------
LASD = r"C:\ArcProj\AboveGroundBiomass\Wait_LasDataset.lasd"
GDB  = r"C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass.gdb"
CHM  = os.path.join(GDB, "chm_raw_CopyRaster")
exists_or_fail(CHM, "CHM raster")

# -----------------------------------------------------------------------------
# AOI (HUGE SPEED WIN) — set to None to disable
# -----------------------------------------------------------------------------
AOI = None

# -----------------------------------------------------------------------------
# PASS THRESHOLDS (Big crowns + still many crowns)
# -----------------------------------------------------------------------------
# Robust thresholds (recommended): use P95 + Max rather than mean.
P95_PERIM_MAX = 80.0     # m
MAX_PERIM_MAX = 150.0    # m
P95_AREA_MAX  = 300.0    # m²
MAX_AREA_MAX  = 700.0    # m²

def pass_rules(perim_stats, area_stats):
    return (
        perim_stats["p95"] <= P95_PERIM_MAX and
        perim_stats["max"] <= MAX_PERIM_MAX and
        area_stats["p95"]  <= P95_AREA_MAX and
        area_stats["max"]  <= MAX_AREA_MAX
    )

# -----------------------------------------------------------------------------
# SEED DENSITY GATE (approx, cheap early reject)
# -----------------------------------------------------------------------------
SEED_RATIO_MIN = 0.00005
SEED_RATIO_MAX = 0.05
GATE_RES_M = 10

# -----------------------------------------------------------------------------
# OUTPUT / PERFORMANCE SETTINGS
# -----------------------------------------------------------------------------
KEEP_ALL_OUTPUTS = False
DO_POLY_SMOOTH = True
PAEK_TOL = "3 Meters"

# For accurate max & P95, keep this 0 (exact). If too slow, set e.g. 50000.
SAMPLE_MAX_FEATURES = 0

REPORT_DIR = r"C:\ArcProj\AboveGroundBiomass\reports"
REPORT_HTML = os.path.join(REPORT_DIR, f"crown_sweep_big_many_{time.strftime('%Y%m%d_%H%M%S')}.html")

RESULTS_TABLE_NAME = safe_name(f"crown_sweep_big_many_{time.strftime('%Y%m%d_%H%M%S')}")
RESULTS_TABLE = ensure_results_table(GDB, RESULTS_TABLE_NAME)

# -----------------------------------------------------------------------------
# PARAMETER SETS (small + targeted to keep bigger crowns, but still allow many)
# -----------------------------------------------------------------------------
# Notes:
# - FMAX up suppresses micro-peaks (reduces fragmentation, keeps bigger crowns)
# - EPS moderate-to-higher reduces over-splitting driven by tiny height differences
# - SMOOTH 1–3 reduces micro-basins while still keeping structure
# Keep this list short for <30 minutes.
PARAM_SETS = [
    #  current best (baseline)
    (1, 1.3, 3,11, 0.8),

    # Capture more crown edge (lower HMIN)
    (1, 1.3, 2, 10, 0.6),
    (1, 1.3, 2, 10, 0.7),

    # Lower HMIN but reduce micro-splitting
    (1, 1.1, 3, 10, 0.6),
    (1, 1.1, 3, 10, 0.7),

    # Lower HMIN + slightly more merging tolerance
    (1, 1.1, 2, 11, 0.6),
    (1, 1.1, 2, 11, 0.7),
    (1, 1.1, 2, 10, 0.6),
    (1, 1.1, 2, 10, 0.7)
]


# -----------------------------------------------------------------------------
# PRE-FLIGHT: CHM sanity check
# -----------------------------------------------------------------------------
mn, mx = raster_minmax(CHM)
stamp(f"CHM range: min={mn}, max={mx}")
if isinstance(mx, (int, float)) and mx > 80:
    stamp("WARNING: CHM max > 80 — double-check this is a true CHM (DSM-DTM).")

# -----------------------------------------------------------------------------
# AOI speed-up
# -----------------------------------------------------------------------------
if AOI:
    exists_or_fail(AOI, "AOI polygon")
    arcpy.env.extent = AOI
    arcpy.env.mask = AOI
    stamp(f"AOI enabled: {AOI}")
else:
    arcpy.env.extent = None
    arcpy.env.mask = None

# -----------------------------------------------------------------------------
# CHM resample cache (per CELL)
# -----------------------------------------------------------------------------
resample_cache = {}

def get_chm_for_cell(cell_size: float):
    try:
        csx = float(arcpy.management.GetRasterProperties(CHM, "CELLSIZEX")[0])
    except:
        csx = None

    if csx is not None and abs(float(cell_size) - csx) < 1e-6:
        return CHM

    if cell_size in resample_cache and arcpy.Exists(resample_cache[cell_size]):
        return resample_cache[cell_size]

    out_ras = os.path.join(GDB, safe_name(f"CHM_cell{cell_size}m"))
    if arcpy.Exists(out_ras):
        resample_cache[cell_size] = out_ras
        return out_ras

    stamp(f"Resampling CHM to {cell_size} m...")
    arcpy.management.Resample(CHM, out_ras, f"{cell_size} {cell_size}", "BILINEAR")
    exists_or_fail(out_ras, f"CHM resampled {cell_size}m")
    resample_cache[cell_size] = out_ras
    return out_ras

# -----------------------------------------------------------------------------
# Raster caches (avoid recomputing heavy rasters)
# -----------------------------------------------------------------------------
cache_canopy = {}   # (CELL,HMIN) -> raster path
cache_smooth = {}   # (CELL,HMIN,SMOOTH) -> raster path
cache_fmax   = {}   # (CELL,HMIN,SMOOTH,FMAX) -> raster path

# -----------------------------------------------------------------------------
# Seed gate (robust approx using coarse resample + numpy)
# -----------------------------------------------------------------------------
def approx_seed_ratio(seed_ras: str, canopy_ras: str, tag: str):
    seed_coarse = os.path.join(GDB, safe_name(f"seed_gate_{tag}"))
    cany_coarse = os.path.join(GDB, safe_name(f"can_gate_{tag}"))

    for p in [seed_coarse, cany_coarse]:
        if arcpy.Exists(p):
            arcpy.management.Delete(p)

    arcpy.management.Resample(seed_ras, seed_coarse, f"{GATE_RES_M} {GATE_RES_M}", "NEAREST")
    arcpy.management.Resample(canopy_ras, cany_coarse, f"{GATE_RES_M} {GATE_RES_M}", "NEAREST")
    exists_or_fail(seed_coarse, "seed_coarse")
    exists_or_fail(cany_coarse, "canopy_coarse")

    seed_r = arcpy.Raster(seed_coarse)
    cany_r = arcpy.Raster(cany_coarse)

    seed_nodata = seed_r.noDataValue
    cany_nodata = cany_r.noDataValue

    a_seed = arcpy.RasterToNumPyArray(seed_r, nodata_to_value=seed_nodata)
    a_cany = arcpy.RasterToNumPyArray(cany_r, nodata_to_value=cany_nodata)

    canopy_cells = int(np.count_nonzero(a_cany != cany_nodata))
    seed_cells   = int(np.count_nonzero(a_seed == 1))

    for p in [seed_coarse, cany_coarse]:
        if arcpy.Exists(p):
            arcpy.management.Delete(p)

    return seed_cells / max(canopy_cells, 1)

# -----------------------------------------------------------------------------
# Run bookkeeping and best-selection logic
# -----------------------------------------------------------------------------
run_rows = []
best_row = None       # best overall (PASS wins; otherwise closest by penalty)
best_pass_row = None  # BEST PASS for your goal: MAX n among PASS (then lower P95 tail)

def penalty(perim_stats, area_stats):
    # Lower is better; 0 means PASS.
    def over(x, lim): return max(0.0, x - lim)
    return (
        over(perim_stats["p95"], P95_PERIM_MAX) * 2.0 +
        over(perim_stats["max"], MAX_PERIM_MAX) * 3.0 +
        over(area_stats["p95"],  P95_AREA_MAX)  * 2.0 +
        over(area_stats["max"],  MAX_AREA_MAX)  * 3.0
    )

def is_better_pass(a, b):
    """
    PASS ranking for goal = big crowns + still many crowns:
    1) maximize n
    2) minimize P95 perimeter (less blob tail)
    3) minimize MAX perimeter
    4) minimize P95 area
    """
    if b is None:
        return True
    if a["n"] != b["n"]:
        return a["n"] > b["n"]
    if a["perim"]["p95"] != b["perim"]["p95"]:
        return a["perim"]["p95"] < b["perim"]["p95"]
    if a["perim"]["max"] != b["perim"]["max"]:
        return a["perim"]["max"] < b["perim"]["max"]
    return a["area"]["p95"] < b["area"]["p95"]

# -----------------------------------------------------------------------------
# One combo runner
# -----------------------------------------------------------------------------
def run_one_combo(CELL, HMIN, SMOOTH, FMAX, EPS, tag="RUN"):
    t0 = time.time()
    status = "OK"
    out_fc = ""
    seed_ratio = 0.0
    errmsg = ""

    CHM_BASE = get_chm_for_cell(CELL)

    # 1) CANOPY (cached)
    k1 = (CELL, HMIN)
    CHM_CANOPY = cache_canopy.get(k1)
    if not CHM_CANOPY or not arcpy.Exists(CHM_CANOPY):
        CHM_CANOPY = os.path.join(GDB, safe_name(f"CHM_CANOPY_c{CELL}_h{HMIN}"))
        if arcpy.Exists(CHM_CANOPY):
            arcpy.management.Delete(CHM_CANOPY)
        SetNull(Raster(CHM_BASE) < HMIN, Raster(CHM_BASE)).save(CHM_CANOPY)
        exists_or_fail(CHM_CANOPY, "CHM_CANOPY")
        cache_canopy[k1] = CHM_CANOPY

    # Lock env to canopy only (stability + speed)
    arcpy.env.snapRaster = CHM_BASE
    arcpy.env.cellSize   = CHM_BASE
    arcpy.env.extent     = CHM_CANOPY
    arcpy.env.mask       = CHM_CANOPY

    # 2) SMOOTH (cached)
    k2 = (CELL, HMIN, SMOOTH)
    CHM_S = cache_smooth.get(k2)
    if not CHM_S or not arcpy.Exists(CHM_S):
        CHM_S = os.path.join(GDB, safe_name(f"CHM_S_c{CELL}_h{HMIN}_s{SMOOTH}"))
        if arcpy.Exists(CHM_S):
            arcpy.management.Delete(CHM_S)
        if SMOOTH == 0:
            Raster(CHM_CANOPY).save(CHM_S)
        else:
            FocalStatistics(Raster(CHM_CANOPY), NbrCircle(SMOOTH, "CELL"), "MEAN", "DATA").save(CHM_S)
        exists_or_fail(CHM_S, "CHM_S")
        cache_smooth[k2] = CHM_S

    # 3) Local max surface (cached)
    k3 = (CELL, HMIN, SMOOTH, FMAX)
    CHM_F = cache_fmax.get(k3)
    if not CHM_F or not arcpy.Exists(CHM_F):
        CHM_F = os.path.join(GDB, safe_name(f"CHM_FMAX_c{CELL}_h{HMIN}_s{SMOOTH}_f{FMAX}"))
        if arcpy.Exists(CHM_F):
            arcpy.management.Delete(CHM_F)
        FocalStatistics(Raster(CHM_S), NbrCircle(FMAX, "CELL"), "MAXIMUM", "DATA").save(CHM_F)
        exists_or_fail(CHM_F, "CHM_FMAX")
        cache_fmax[k3] = CHM_F

    combo_tag = safe_name(f"{tag}_c{CELL}_h{HMIN}_s{SMOOTH}_f{FMAX}_e{EPS}")
    SEEDS_RAS  = os.path.join(GDB, safe_name(f"SEEDS_{combo_tag}"))
    WS_RAS     = os.path.join(GDB, safe_name(f"WS_{combo_tag}"))
    WS_CLEAN   = os.path.join(GDB, safe_name(f"WSCL_{combo_tag}"))
    CROWNS_RAW = os.path.join(GDB, safe_name(f"CROWNS_{combo_tag}"))
    CROWNS_FIN = os.path.join(GDB, safe_name(f"CROWNSF_{combo_tag}"))

    for p in [SEEDS_RAS, WS_RAS, WS_CLEAN, CROWNS_RAW, CROWNS_FIN]:
        if arcpy.Exists(p):
            arcpy.management.Delete(p)

    try:
        # 4) Seeds (Thin OFF by design)
        SEEDS = Con(
            (Raster(CHM_S) >= (Raster(CHM_F) - EPS)) &
            (Raster(CHM_S) >= HMIN),
            1
        )
        SEEDS.save(SEEDS_RAS)
        exists_or_fail(SEEDS_RAS, "SEEDS_RAS")

        # Seed gate
        seed_ratio = approx_seed_ratio(SEEDS_RAS, CHM_CANOPY, tag=combo_tag)
        if seed_ratio < SEED_RATIO_MIN:
            status = "SKIP_TOO_FEW_SEEDS"
            raise RuntimeError(status)
        if seed_ratio > SEED_RATIO_MAX:
            status = "SKIP_TOO_MANY_SEEDS"
            raise RuntimeError(status)

        # 5) Watershed
        INV = -1 * Raster(CHM_S)
        FDR = FlowDirection(INV, "FORCE")
        Watershed(FDR, SEEDS_RAS).save(WS_RAS)
        exists_or_fail(WS_RAS, "WS_RAS")

        # 5b) Reduce tiny zones before polygonizing
        MajorityFilter(Raster(WS_RAS), "EIGHT", "HALF").save(WS_CLEAN)
        exists_or_fail(WS_CLEAN, "WS_CLEAN")

        # 6) Polygonize
        arcpy.conversion.RasterToPolygon(
            in_raster=WS_CLEAN,
            out_polygon_features=CROWNS_RAW,
            simplify="SIMPLIFY",
            raster_field="Value",
            create_multipart_features="SINGLE_OUTER_PART"
        )
        exists_or_fail(CROWNS_RAW, "CROWNS_RAW")

        # 7) Optional smoothing
        if DO_POLY_SMOOTH:
            arcpy.cartography.SmoothPolygon(
                in_features=CROWNS_RAW,
                out_feature_class=CROWNS_FIN,
                algorithm="PAEK",
                tolerance=PAEK_TOL
            )
            exists_or_fail(CROWNS_FIN, "CROWNS_FIN")
            out_fc = CROWNS_FIN
        else:
            out_fc = CROWNS_RAW

        # 8) Stats
        n = int(arcpy.management.GetCount(out_fc)[0])
        if n == 0:
            status = "FAIL_EMPTY"
            perims, areas = [], []
        else:
            perims, areas = collect_perim_and_area(out_fc, sample_max_features=SAMPLE_MAX_FEATURES)

        perim_stats = summarize(perims)
        area_stats  = summarize(areas)
        dt = time.time() - t0

        pass_flag = "PASS" if (n > 0 and pass_rules(perim_stats, area_stats)) else "FAIL"
        pen = penalty(perim_stats, area_stats) if n > 0 else 1e9

        row = {
            "CELL": CELL, "HMIN": HMIN, "SMOOTH": SMOOTH, "FMAX": FMAX, "EPS": EPS,
            "status": status, "seconds": dt,
            "seed_ratio": seed_ratio, "errmsg": "",
            "n": int(n),
            "perim": perim_stats,
            "area": area_stats,
            "pass_flag": pass_flag,
            "penalty": pen,
            "out_fc": out_fc
        }

        log_result(
            RESULTS_TABLE,
            (CELL, HMIN, SMOOTH, FMAX, EPS,
             float(seed_ratio),
             int(n),
             float(perim_stats["mean"]), float(perim_stats["p95"]), float(perim_stats["max"]),
             float(area_stats["mean"]),  float(area_stats["p95"]),  float(area_stats["max"]),
             pass_flag,
             float(dt),
             out_fc,
             status,
             "")
        )

        if not KEEP_ALL_OUTPUTS:
            for p in [SEEDS_RAS, WS_RAS, WS_CLEAN, CROWNS_RAW]:
                if arcpy.Exists(p) and p != out_fc:
                    arcpy.management.Delete(p)

        return row

    except Exception as e:
        dt = time.time() - t0
        if not status.startswith("SKIP"):
            status = "ERROR"
        errmsg = (str(e) or "unknown error")[:255]

        try:
            log_result(
                RESULTS_TABLE,
                (CELL, HMIN, SMOOTH, FMAX, EPS,
                 float(seed_ratio),
                 0,
                 0.0, 0.0, 0.0,
                 0.0, 0.0, 0.0,
                 "FAIL",
                 float(dt),
                 "",
                 status,
                 errmsg)
            )
        except:
            pass

        for p in [SEEDS_RAS, WS_RAS, WS_CLEAN, CROWNS_RAW, CROWNS_FIN]:
            if arcpy.Exists(p):
                arcpy.management.Delete(p)

        return {
            "CELL": CELL, "HMIN": HMIN, "SMOOTH": SMOOTH, "FMAX": FMAX, "EPS": EPS,
            "status": status, "seconds": dt,
            "seed_ratio": seed_ratio, "errmsg": errmsg,
            "n": 0,
            "perim": summarize([]),
            "area": summarize([]),
            "pass_flag": "FAIL",
            "penalty": 1e9,
            "out_fc": ""
        }

# -----------------------------------------------------------------------------
# MAIN
# -----------------------------------------------------------------------------
try:
    stamp(f"Running {len(PARAM_SETS)} targeted combos (big crowns + many crowns)...")

    for (CELL, HMIN, SMOOTH, FMAX, EPS) in tqdm(PARAM_SETS, desc="Sweep", unit="combo"):
        row = run_one_combo(CELL, HMIN, SMOOTH, FMAX, EPS, tag="LAST_1m")
        run_rows.append(row)

        # Track best PASS: max n among PASS, then smaller tail
        if row["status"] == "OK" and row["pass_flag"] == "PASS" and row["n"] > 0:
            if is_better_pass(row, best_pass_row):
                best_pass_row = row

        # Track best overall if nothing passes: minimum penalty
        if row["status"] == "OK" and row["n"] > 0:
            if (best_row is None) or (row["penalty"] < best_row["penalty"]):
                best_row = row

except Exception:
    stamp("FATAL ERROR (full traceback):")
    print(traceback.format_exc())

finally:
    winner = best_pass_row if best_pass_row is not None else best_row

    meta = {
        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "GDB": GDB,
        "CHM": CHM,
        "AOI": AOI,
        "P95_PERIM_MAX": P95_PERIM_MAX,
        "MAX_PERIM_MAX": MAX_PERIM_MAX,
        "P95_AREA_MAX": P95_AREA_MAX,
        "MAX_AREA_MAX": MAX_AREA_MAX,
        "SEED_RATIO_MIN": SEED_RATIO_MIN,
        "SEED_RATIO_MAX": SEED_RATIO_MAX,
        "GATE_RES_M": GATE_RES_M,
        "DO_POLY_SMOOTH": DO_POLY_SMOOTH,
        "PAEK_TOL": PAEK_TOL,
        "KEEP_ALL_OUTPUTS": KEEP_ALL_OUTPUTS,
        "SAMPLE_MAX_FEATURES": SAMPLE_MAX_FEATURES,
        "RESULTS_TABLE": RESULTS_TABLE,
        "PARAM_SETS": PARAM_SETS,
        "BEST_RULE": "BEST(PASS)=max n, then min P95 perimeter, then min max perimeter, then min P95 area"
    }

    stamp(f"Writing HTML report: {REPORT_HTML}")
    write_html_report(REPORT_HTML, meta, run_rows, best_row=winner, pass_row=best_pass_row)
    stamp(f"✔ Report written: {REPORT_HTML}")
    stamp(f"Results table: {RESULTS_TABLE}")

    if winner:
        stamp("WINNER:")
        stamp(f"  PASS? {winner['pass_flag']} | status={winner['status']}")
        stamp(f"  Params: CELL={winner['CELL']}, HMIN={winner['HMIN']}, SMOOTH={winner['SMOOTH']}, FMAX={winner['FMAX']}, EPS={winner['EPS']}")
        stamp(f"  n={winner['n']:,} | seed_ratio≈{winner.get('seed_ratio',0):.6f}")
        stamp(f"  Perim mean/p95/max (m): {winner['perim']['mean']:.2f} / {winner['perim']['p95']:.2f} / {winner['perim']['max']:.2f}")
        stamp(f"  Area  mean/p95/max (m²): {winner['area']['mean']:.2f} / {winner['area']['p95']:.2f} / {winner['area']['max']:.2f}")
        stamp(f"  Output FC: {winner['out_fc']}")
    else:
        stamp("No successful crowns were produced. Check ERRMSG in the HTML/table.")

    try:
        import winsound
        winsound.Beep(880, 500)
    except:
        pass


[23:57:58] CHM range: min=-17.8419952392578, max=30.4289970397949
[23:57:58] Running 9 targeted combos (big crowns + many crowns)...


Sweep: 100%|██████████| 9/9 [29:52<00:00, 199.11s/combo]﻿


[00:27:50] Writing HTML report: C:\ArcProj\AboveGroundBiomass\reports\crown_sweep_big_many_20260217_235747.html
[00:27:50] ✔ Report written: C:\ArcProj\AboveGroundBiomass\reports\crown_sweep_big_many_20260217_235747.html
[00:27:50] Results table: C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass.gdb\crown_sweep_big_many_20260217_235747
[00:27:50] WINNER:
[00:27:50]   PASS? FAIL | status=OK
[00:27:50]   Params: CELL=1, HMIN=1.1, SMOOTH=2, FMAX=11, EPS=0.6
[00:27:50]   n=32,803 | seed_ratio≈0.006904
[00:27:50]   Perim mean/p95/max (m): 18.75 / 54.60 / 615.47
[00:27:50]   Area  mean/p95/max (m²): 31.90 / 128.37 / 1993.90
[00:27:50]   Output FC: C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass.gdb\CROWNSF_LAST_1m_c1_h1_1_s2_f11_e0_6


In [2]:
import os
import time
import math
import statistics
import traceback
import arcpy
import numpy as np
from arcpy.sa import *
from tqdm import tqdm

# -------------------------------------------------------------------
# WHAT THIS SCRIPT DOES
# -------------------------------------------------------------------
# Fast crown-parameter tuning for CHM-based crown delineation.
#
# Workflow (unchanged algorithm):
#   canopy mask → smoothing → local-max surface → seed raster → watershed → polygons
#
# Key features:
#  - AOI support (major speed-up) – keeps processing constrained spatially
#  - Raster caching (canopy / smoothed / fmax) so only EPS triggers re-runs
#  - Early gating: skip combos with too few/many seeds before Watershed
#  - Results: writes a GDB results table + an HTML report
#
# IMPORTANT: "best" is NOT "smallest max diameter"
#  - smallest max often means over-splitting into tiny crowns (not what you want)
#  - we prefer: PASS runs (max <= threshold) with high crown count (n),
#               then smaller p95 as a secondary tie-break
# -------------------------------------------------------------------

# =============================================================================
# ENVIRONMENT
# =============================================================================
arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")
arcpy.CheckOutExtension("3D")
arcpy.env.parallelProcessingFactor = "75%"  # best-effort; some tools ignore it

def stamp(msg: str):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

def exists_or_fail(path: str, what: str):
    if not arcpy.Exists(path):
        raise RuntimeError(f"{what} missing: {path}")

def safe_name(s: str) -> str:
    out = "".join(ch if ch.isalnum() else "_" for ch in s)
    return out[:60]

# =============================================================================
# RECOMMENDED TUNING DEFAULTS (your "best level" starting point)
# =============================================================================
# These are the values to try first on the AOI. If under-splitting occurs, raise EPS to 0.15–0.20.
RECOMMENDED = {
    "CELL": 1,
    "HMIN": 2,
    "SMOOTH": 0,
    "FMAX": 6,
    "EPS": 0.10
}

# =============================================================================
# INPUTS
# =============================================================================
GDB = r"C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass.gdb"
CHM = os.path.join(GDB, "chm_raw_CopyRaster")

# AOI is strongly recommended for tuning speed.
AOI = r"C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass.gdb\AOI_poly"  # set to None to disable

exists_or_fail(GDB, "GDB")
exists_or_fail(CHM, "CHM raster")
if AOI:
    exists_or_fail(AOI, "AOI polygon")

# =============================================================================
# STOPPING / PASS RULE
# =============================================================================
# A run is a PASS if max crown equivalent diameter <= this threshold.
# Use 30 as a practical starting point; tighten if you want.
MAX_UK_CROWN_DIAM_M = 30.0

# =============================================================================
# SEED DENSITY GATE (fast sanity check BEFORE watershed)
# =============================================================================
# We compute a coarse-grid approximate seed density: seed_cells / canopy_cells.
# This avoids GetRasterProperties COUNT/SUM (unsupported in your Pro build).
SEED_RATIO_MIN = 0.00005
SEED_RATIO_MAX = 0.05

# Coarse resampling resolution (meters) used only for the seed density gate.
# For your ~0.87 km² AOI, 20–30 m is usually faster and stable.
GATE_RES_M = 20

# =============================================================================
# SWEEP STRATEGY (coarse → refine)
# =============================================================================
# Stage 1 stays close to recommended defaults. Stage 2 expands around best found.
CELL_STAGE1   = [RECOMMENDED["CELL"]]
HMIN_STAGE1   = [RECOMMENDED["HMIN"]]
SMOOTH_STAGE1 = [0, 1, 2]                # small range; smoothing can merge crowns if too high
FMAX_STAGE1   = [5, 6, 8]                # 6 tends to be a good default
EPS_STAGE1    = [0.10, 0.20, 0.35, 0.50, 0.70, 1.00]

# Stage 2 refines around the "best" run found in Stage 1
EPS_OFFSETS   = [-0.20, -0.15, -0.10, -0.05, 0.0, 0.05, 0.10, 0.15, 0.20]
SMOOTH_REFINE = [0, 1, 2, 3, 4]
FMAX_REFINE   = [2, 3, 4, 5, 6, 8]
HMIN_REFINE   = [2.0, 2.5, 3.0, 3.5]
CELL_REFINE   = [1, 2]

COARSE_ONLY = False

# =============================================================================
# OUTPUTS / REPORT
# =============================================================================
# Speed tip: smoothing polygons during the sweep can be expensive.
# Keep OFF during tuning; smooth only final chosen output.
DO_POLY_SMOOTH_SWEEP = False
DO_POLY_SMOOTH_FINAL = True
PAEK_TOL = "3 Meters"

# If you have huge n, sample for stats to avoid slow cursor scans.
SAMPLE_MAX_FEATURES = 50000

# Keep intermediate rasters/FCs? Usually False, saves disk/time.
KEEP_ALL_OUTPUTS = False

REPORT_DIR  = r"C:\ArcProj\AboveGroundBiomass\reports"
RUNSTAMP    = time.strftime("%Y%m%d_%H%M%S")
REPORT_HTML = os.path.join(REPORT_DIR, f"crown_sweep_fast_{RUNSTAMP}.html")

RESULTS_TABLE_NAME = safe_name(f"crown_sweep_{RUNSTAMP}")
RESULTS_TABLE = os.path.join(GDB, RESULTS_TABLE_NAME)

# =============================================================================
# HELPERS
# =============================================================================
def raster_minmax(ras_path: str):
    mn = arcpy.management.GetRasterProperties(ras_path, "MINIMUM")[0]
    mx = arcpy.management.GetRasterProperties(ras_path, "MAXIMUM")[0]
    try:
        return float(mn), float(mx)
    except:
        return mn, mx

def equiv_circle_diam_m(area_m2: float) -> float:
    """Equivalent-circle diameter: diameter of a circle with same area as polygon."""
    if area_m2 <= 0:
        return 0.0
    return 2.0 * math.sqrt(area_m2 / math.pi)

def percentile(sorted_vals, p: float):
    n = len(sorted_vals)
    if n == 0:
        return 0.0
    if n == 1:
        return float(sorted_vals[0])
    r = (p / 100.0) * (n - 1)
    lo = int(math.floor(r))
    hi = int(math.ceil(r))
    if lo == hi:
        return float(sorted_vals[lo])
    w = r - lo
    return float(sorted_vals[lo] * (1 - w) + sorted_vals[hi] * w)

def summarize(values):
    if not values:
        return {"n": 0, "min": 0, "max": 0, "mean": 0, "std": 0,
                "p01": 0, "p05": 0, "p25": 0, "p50": 0, "p75": 0, "p95": 0, "p99": 0}
    vals = sorted(values)
    n = len(vals)
    mean = sum(vals) / n
    std = statistics.pstdev(vals) if n > 1 else 0.0
    return {
        "n": n,
        "min": float(vals[0]),
        "max": float(vals[-1]),
        "mean": float(mean),
        "std": float(std),
        "p01": percentile(vals, 1),
        "p05": percentile(vals, 5),
        "p25": percentile(vals, 25),
        "p50": percentile(vals, 50),
        "p75": percentile(vals, 75),
        "p95": percentile(vals, 95),
        "p99": percentile(vals, 99),
    }

def ensure_results_table(gdb: str, table_name: str):
    tpath = os.path.join(gdb, table_name)
    if arcpy.Exists(tpath):
        return tpath
    arcpy.management.CreateTable(gdb, table_name)
    for fn, ft, fl in [
        ("CELL","DOUBLE",None),
        ("HMIN","DOUBLE",None),
        ("SMOOTH","LONG",None),
        ("FMAX","LONG",None),
        ("EPS","DOUBLE",None),
        ("SEED_RATIO","DOUBLE",None),
        ("CROWN_N","LONG",None),
        ("MAX_DIAM_M","DOUBLE",None),
        ("MEAN_DIAM_M","DOUBLE",None),
        ("P95_DIAM_M","DOUBLE",None),
        ("TOTAL_AREA_M2","DOUBLE",None),
        ("SECONDS","DOUBLE",None),
        ("OUT_FC","TEXT",255),
        ("STATUS","TEXT",40),
        ("ERRMSG","TEXT",255),
    ]:
        if ft == "TEXT":
            arcpy.management.AddField(tpath, fn, ft, field_length=fl)
        else:
            arcpy.management.AddField(tpath, fn, ft)
    return tpath

def log_result(table_path, row):
    fields = [
        "CELL","HMIN","SMOOTH","FMAX","EPS",
        "SEED_RATIO","CROWN_N","MAX_DIAM_M","MEAN_DIAM_M","P95_DIAM_M",
        "TOTAL_AREA_M2","SECONDS","OUT_FC","STATUS","ERRMSG"
    ]
    with arcpy.da.InsertCursor(table_path, fields) as ic:
        ic.insertRow(row)

def collect_area_and_diam(fc: str, sample_max_features: int = 0):
    """Collect polygon areas + equivalent diameters; optionally subsample for speed."""
    areas, diams = [], []
    n = int(arcpy.management.GetCount(fc)[0])
    if n == 0:
        return areas, diams

    step = 1
    if sample_max_features and n > sample_max_features:
        step = max(1, n // sample_max_features)

    i = 0
    with arcpy.da.SearchCursor(fc, ["SHAPE@AREA"]) as cur:
        for (a,) in cur:
            if step > 1 and (i % step) != 0:
                i += 1
                continue
            aa = float(a)
            areas.append(aa)
            diams.append(equiv_circle_diam_m(aa))
            i += 1
    return areas, diams

def write_html_report(html_path, meta, run_rows, best_row=None, pass_row=None):
    def esc(s):
        s = "" if s is None else str(s)
        return (s.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
                 .replace('"',"&quot;").replace("'","&#39;"))
    def fmt(x, nd=2):
        try:
            return f"{float(x):.{nd}f}"
        except:
            return esc(x)

    lines = []
    lines.append("<!DOCTYPE html><html><head><meta charset='utf-8'>")
    lines.append("<title>Crown sweep report</title>")
    lines.append("""
<style>
body{font-family:Arial,Helvetica,sans-serif;margin:20px;line-height:1.35}
.small{color:#555;font-size:0.95em}
table{border-collapse:collapse;width:100%;margin:12px 0}
th,td{border:1px solid #ddd;padding:6px 8px;font-size:0.92em;vertical-align:top}
th{background:#f6f6f6;text-align:left}
.good{background:#ecffef}
.bad{background:#ffecec}
.mono{font-family:ui-monospace, SFMono-Regular, Menlo, Consolas, monospace}
</style>
""")
    lines.append("</head><body>")
    lines.append("<h1>Crown parameter sweep report</h1>")
    lines.append(f"<div class='small'>Generated: {esc(meta['generated_at'])}</div>")

    lines.append("<h2>Inputs & rules</h2><table>")
    for k in meta.keys():
        lines.append(f"<tr><th>{esc(k)}</th><td class='mono'>{esc(meta.get(k))}</td></tr>")
    lines.append("</table>")

    lines.append("<h2>Recommended start</h2>")
    lines.append(f"<p class='mono'>CELL={RECOMMENDED['CELL']}, HMIN={RECOMMENDED['HMIN']}, SMOOTH={RECOMMENDED['SMOOTH']}, "
                 f"FMAX={RECOMMENDED['FMAX']}, EPS={RECOMMENDED['EPS']}</p>")
    lines.append("<p class='small'>If under-splitting: raise EPS to 0.15–0.20. If blobs: lower EPS and/or raise HMIN.</p>")

    lines.append("<h2>Highlights</h2>")
    if best_row:
        lines.append("<p><b>Best run (PASS-first, then highest n, then smallest p95)</b></p>")
        lines.append("<table>")
        lines.append(f"<tr><th>Params</th><td class='mono'>CELL={best_row['CELL']}, HMIN={best_row['HMIN']}, SMOOTH={best_row['SMOOTH']}, FMAX={best_row['FMAX']}, EPS={best_row['EPS']}</td></tr>")
        lines.append(f"<tr><th>Status</th><td>{esc(best_row.get('status'))}</td></tr>")
        lines.append(f"<tr><th>Seed ratio</th><td>{fmt(best_row.get('seed_ratio'),6)}</td></tr>")
        lines.append(f"<tr><th>n</th><td>{best_row['n']:,}</td></tr>")
        lines.append(f"<tr><th>Max diam (m)</th><td>{fmt(best_row['diam']['max'])}</td></tr>")
        lines.append(f"<tr><th>P95 diam (m)</th><td>{fmt(best_row['diam']['p95'])}</td></tr>")
        lines.append(f"<tr><th>Mean diam (m)</th><td>{fmt(best_row['diam']['mean'])}</td></tr>")
        lines.append(f"<tr><th>Output FC</th><td class='mono'>{esc(best_row['out_fc'])}</td></tr>")
        lines.append("</table>")
    else:
        lines.append("<p>No successful run produced crowns.</p>")

    if pass_row:
        lines.append("<p><b>First PASS run encountered</b></p>")
        lines.append("<table>")
        lines.append(f"<tr><th>Params</th><td class='mono'>CELL={pass_row['CELL']}, HMIN={pass_row['HMIN']}, SMOOTH={pass_row['SMOOTH']}, FMAX={pass_row['FMAX']}, EPS={pass_row['EPS']}</td></tr>")
        lines.append(f"<tr><th>Max diam (m)</th><td>{fmt(pass_row['diam']['max'])} (≤ {fmt(meta['MAX_UK_CROWN_DIAM_M'])})</td></tr>")
        lines.append(f"<tr><th>Output FC</th><td class='mono'>{esc(pass_row['out_fc'])}</td></tr>")
        lines.append("</table>")

    lines.append("<h2>All runs</h2>")
    lines.append("<div class='small'>Rows are in run order. “SKIP_*” = gated before Watershed. ERRMSG truncated.</div>")
    lines.append("<table>")
    headers = ["CELL","HMIN","SMOOTH","FMAX","EPS","status","seconds","seed_ratio","n","diam_max","diam_mean","diam_p95","diam_p99","out_fc","errmsg"]
    lines.append("<tr>" + "".join(f"<th>{h}</th>" for h in headers) + "</tr>")
    for r in run_rows:
        is_ok = (r.get("status") == "OK" and r.get("n", 0) > 0)
        is_pass = is_ok and (r["diam"]["max"] <= meta["MAX_UK_CROWN_DIAM_M"])
        cls = "good" if is_pass else ("bad" if r.get("status","").startswith("ERROR") else "")
        lines.append(f"<tr class='{cls}'>" + "".join([
            f"<td>{esc(r.get('CELL'))}</td>",
            f"<td>{esc(r.get('HMIN'))}</td>",
            f"<td>{esc(r.get('SMOOTH'))}</td>",
            f"<td>{esc(r.get('FMAX'))}</td>",
            f"<td>{esc(r.get('EPS'))}</td>",
            f"<td>{esc(r.get('status'))}</td>",
            f"<td>{fmt(r.get('seconds',0),1)}</td>",
            f"<td>{fmt(r.get('seed_ratio',0),6)}</td>",
            f"<td>{int(r.get('n',0)):,}</td>",
            f"<td>{fmt(r.get('diam',{}).get('max',0))}</td>",
            f"<td>{fmt(r.get('diam',{}).get('mean',0))}</td>",
            f"<td>{fmt(r.get('diam',{}).get('p95',0))}</td>",
            f"<td>{fmt(r.get('diam',{}).get('p99',0))}</td>",
            f"<td class='mono'>{esc(r.get('out_fc',''))}</td>",
            f"<td class='mono'>{esc(r.get('errmsg',''))}</td>",
        ]) + "</tr>")
    lines.append("</table></body></html>")

    os.makedirs(os.path.dirname(html_path), exist_ok=True)
    with open(html_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

# =============================================================================
# "BEST IS NOT SMALLEST MAX" COMPARATOR
# =============================================================================
def is_better(a, b, max_diam_thresh):
    """
    Preference order:
      1) Prefer OK runs over non-OK.
      2) Prefer PASS runs (max <= threshold) over non-PASS.
      3) Among PASS runs: higher n (more crowns) is better.
      4) Tie-break among PASS: smaller p95 is better (keeps upper tail sensible).
      5) If neither PASS: smaller max is better, then smaller p99.
    """
    if b is None:
        return True

    a_ok = (a["status"] == "OK" and a["n"] > 0)
    b_ok = (b["status"] == "OK" and b["n"] > 0)

    if a_ok and not b_ok:
        return True
    if b_ok and not a_ok:
        return False
    if not a_ok and not b_ok:
        # both not-ok: prefer smaller max (least-bad)
        return a["diam"]["max"] < b["diam"]["max"]

    a_pass = (a["diam"]["max"] <= max_diam_thresh)
    b_pass = (b["diam"]["max"] <= max_diam_thresh)

    if a_pass and not b_pass:
        return True
    if b_pass and not a_pass:
        return False

    if a_pass and b_pass:
        if a["n"] != b["n"]:
            return a["n"] > b["n"]
        return a["diam"]["p95"] < b["diam"]["p95"]

    # neither pass
    if a["diam"]["max"] != b["diam"]["max"]:
        return a["diam"]["max"] < b["diam"]["max"]
    return a["diam"]["p99"] < b["diam"]["p99"]

# =============================================================================
# AOI ENVIRONMENT SETUP
# =============================================================================
AOI_USED = False
if AOI:
    arcpy.env.extent = AOI
    arcpy.env.mask   = AOI
    AOI_USED = True
    stamp(f"AOI enabled: {AOI}")
else:
    arcpy.env.extent = None
    arcpy.env.mask   = None

# Pre-flight CHM check
mn, mx = raster_minmax(CHM)
stamp(f"CHM range: min={mn}, max={mx}")
if isinstance(mx, (int, float)) and mx > 80:
    stamp("WARNING: CHM max > 80 — double-check this is a true canopy height model (DSM-DTM).")

# Create results table
RESULTS_TABLE = ensure_results_table(GDB, RESULTS_TABLE_NAME)

# =============================================================================
# CHM RESAMPLE CACHE (per CELL)
# =============================================================================
resample_cache = {}

def get_chm_for_cell(cell_size: float):
    """
    If CHM native cell size already matches, return CHM.
    Otherwise resample once per requested cell size and cache.
    """
    try:
        csx = float(arcpy.management.GetRasterProperties(CHM, "CELLSIZEX")[0])
    except:
        csx = None

    if csx is not None and abs(float(cell_size) - csx) < 1e-6:
        return CHM

    if cell_size in resample_cache and arcpy.Exists(resample_cache[cell_size]):
        return resample_cache[cell_size]

    out_ras = os.path.join(GDB, safe_name(f"CHM_cell{cell_size}m_{RUNSTAMP}"))
    if arcpy.Exists(out_ras):
        resample_cache[cell_size] = out_ras
        return out_ras

    stamp(f"Resampling CHM to {cell_size} m...")
    arcpy.management.Resample(CHM, out_ras, f"{cell_size} {cell_size}", "BILINEAR")
    exists_or_fail(out_ras, f"CHM resampled {cell_size}m")
    resample_cache[cell_size] = out_ras
    return out_ras

# =============================================================================
# RASTER CACHES
# =============================================================================
cache_canopy = {}   # (CELL,HMIN) -> raster path
cache_smooth = {}   # (CELL,HMIN,SMOOTH) -> raster path
cache_fmax   = {}   # (CELL,HMIN,SMOOTH,FMAX) -> raster path

# =============================================================================
# SEED GATE (coarse resample + numpy)
# =============================================================================
def approx_seed_ratio(seed_ras: str, canopy_ras: str, tag: str):
    """
    Returns approx seed_ratio = seed_cells / canopy_cells computed on a coarsened grid.
    NEAREST resampling preserves seed 1-cells as 1 (or NoData).
    """
    seed_coarse = os.path.join(GDB, safe_name(f"seed_gate_{tag}_{RUNSTAMP}"))
    cany_coarse = os.path.join(GDB, safe_name(f"can_gate_{tag}_{RUNSTAMP}"))

    for p in [seed_coarse, cany_coarse]:
        if arcpy.Exists(p):
            arcpy.management.Delete(p)

    arcpy.management.Resample(seed_ras, seed_coarse, f"{GATE_RES_M} {GATE_RES_M}", "NEAREST")
    arcpy.management.Resample(canopy_ras, cany_coarse, f"{GATE_RES_M} {GATE_RES_M}", "NEAREST")
    exists_or_fail(seed_coarse, "seed_coarse")
    exists_or_fail(cany_coarse, "canopy_coarse")

    seed_r = arcpy.Raster(seed_coarse)
    cany_r = arcpy.Raster(cany_coarse)

    seed_nodata = seed_r.noDataValue
    cany_nodata = cany_r.noDataValue

    a_seed = arcpy.RasterToNumPyArray(seed_r, nodata_to_value=seed_nodata)
    a_cany = arcpy.RasterToNumPyArray(cany_r, nodata_to_value=cany_nodata)

    canopy_cells = int(np.count_nonzero(a_cany != cany_nodata))
    seed_cells   = int(np.count_nonzero(a_seed == 1))

    # cleanup coarse rasters
    for p in [seed_coarse, cany_coarse]:
        if arcpy.Exists(p):
            arcpy.management.Delete(p)

    return seed_cells / max(canopy_cells, 1)

# =============================================================================
# RUN ONE PARAM COMBINATION
# =============================================================================
def run_one_combo(CELL, HMIN, SMOOTH, FMAX, EPS, stage_tag="S1"):
    """
    Runs the full algorithm for one parameter set, using cached rasters where possible.
    Returns a dict with stats/status.
    """
    t0 = time.time()
    status = "OK"
    out_fc = ""
    seed_ratio = 0.0
    errmsg = ""

    CHM_BASE = get_chm_for_cell(CELL)

    # 1) CANOPY MASK (cached by CELL,HMIN)
    k1 = (CELL, HMIN)
    CHM_CANOPY = cache_canopy.get(k1)
    if not CHM_CANOPY or not arcpy.Exists(CHM_CANOPY):
        CHM_CANOPY = os.path.join(GDB, safe_name(f"CHM_CANOPY_c{CELL}_h{HMIN}_{RUNSTAMP}"))
        if arcpy.Exists(CHM_CANOPY):
            arcpy.management.Delete(CHM_CANOPY)
        SetNull(Raster(CHM_BASE) < HMIN, Raster(CHM_BASE)).save(CHM_CANOPY)
        exists_or_fail(CHM_CANOPY, "CHM_CANOPY")
        cache_canopy[k1] = CHM_CANOPY

    # Lock env:
    # - extent should remain AOI if provided (do NOT overwrite with canopy extent)
    arcpy.env.snapRaster = CHM_BASE
    arcpy.env.cellSize   = CHM_BASE
    arcpy.env.extent     = AOI if AOI_USED else CHM_CANOPY
    arcpy.env.mask       = CHM_CANOPY

    # 2) SMOOTH (cached by CELL,HMIN,SMOOTH)
    k2 = (CELL, HMIN, SMOOTH)
    CHM_S = cache_smooth.get(k2)
    if not CHM_S or not arcpy.Exists(CHM_S):
        CHM_S = os.path.join(GDB, safe_name(f"CHM_S_c{CELL}_h{HMIN}_s{SMOOTH}_{RUNSTAMP}"))
        if arcpy.Exists(CHM_S):
            arcpy.management.Delete(CHM_S)
        if SMOOTH == 0:
            Raster(CHM_CANOPY).save(CHM_S)
        else:
            FocalStatistics(Raster(CHM_CANOPY), NbrCircle(SMOOTH, "CELL"), "MEAN", "DATA").save(CHM_S)
        exists_or_fail(CHM_S, "CHM_S")
        cache_smooth[k2] = CHM_S

    # 3) LOCAL MAX SURFACE (cached by CELL,HMIN,SMOOTH,FMAX)
    k3 = (CELL, HMIN, SMOOTH, FMAX)
    CHM_F = cache_fmax.get(k3)
    if not CHM_F or not arcpy.Exists(CHM_F):
        CHM_F = os.path.join(GDB, safe_name(f"CHM_FMAX_c{CELL}_h{HMIN}_s{SMOOTH}_f{FMAX}_{RUNSTAMP}"))
        if arcpy.Exists(CHM_F):
            arcpy.management.Delete(CHM_F)
        FocalStatistics(Raster(CHM_S), NbrCircle(FMAX, "CELL"), "MAXIMUM", "DATA").save(CHM_F)
        exists_or_fail(CHM_F, "CHM_FMAX")
        cache_fmax[k3] = CHM_F

    # EPS varies per run → unique names per run
    combo_tag = safe_name(f"{stage_tag}_c{CELL}_h{HMIN}_s{SMOOTH}_f{FMAX}_e{EPS}_{RUNSTAMP}")
    SEEDS_RAS  = os.path.join(GDB, safe_name(f"SEEDS_{combo_tag}"))
    WS_RAS     = os.path.join(GDB, safe_name(f"WS_{combo_tag}"))
    WS_CLEAN   = os.path.join(GDB, safe_name(f"WSCL_{combo_tag}"))
    CROWNS_RAW = os.path.join(GDB, safe_name(f"CROWNS_{combo_tag}"))
    CROWNS_FIN = os.path.join(GDB, safe_name(f"CROWNSF_{combo_tag}"))

    # Clean outputs for this combo
    for p in [SEEDS_RAS, WS_RAS, WS_CLEAN, CROWNS_RAW, CROWNS_FIN]:
        if arcpy.Exists(p):
            arcpy.management.Delete(p)

    try:
        # 4) SEEDS (Thin is OFF)
        # A cell is a seed if it is within EPS meters of the local maximum in its FMAX window.
        SEEDS = Con(
            (Raster(CHM_S) >= (Raster(CHM_F) - EPS)) &
            (Raster(CHM_S) >= HMIN),
            1
        )
        SEEDS.save(SEEDS_RAS)
        exists_or_fail(SEEDS_RAS, "SEEDS_RAS")

        # 4b) Seed density gate (cheap check before expensive watershed)
        seed_ratio = approx_seed_ratio(SEEDS_RAS, CHM_CANOPY, tag=combo_tag)
        if seed_ratio < SEED_RATIO_MIN:
            status = "SKIP_TOO_FEW_SEEDS"
            raise RuntimeError(status)
        if seed_ratio > SEED_RATIO_MAX:
            status = "SKIP_TOO_MANY_SEEDS"
            raise RuntimeError(status)

        # 5) WATERSHED
        INV = -1 * Raster(CHM_S)
        FDR = FlowDirection(INV, "FORCE")
        Watershed(FDR, SEEDS_RAS).save(WS_RAS)
        exists_or_fail(WS_RAS, "WS_RAS")

        # 5b) CLEAN (reduces tiny zones → faster polygonise)
        MajorityFilter(Raster(WS_RAS), "EIGHT", "HALF").save(WS_CLEAN)
        exists_or_fail(WS_CLEAN, "WS_CLEAN")

        # 6) Polygonize
        arcpy.conversion.RasterToPolygon(
            in_raster=WS_CLEAN,
            out_polygon_features=CROWNS_RAW,
            simplify="SIMPLIFY",
            raster_field="Value",
            create_multipart_features="SINGLE_OUTER_PART"
        )
        exists_or_fail(CROWNS_RAW, "CROWNS_RAW")

        # 7) Optional polygon smoothing (OFF during sweep by default)
        if DO_POLY_SMOOTH_SWEEP:
            arcpy.cartography.SmoothPolygon(
                in_features=CROWNS_RAW,
                out_feature_class=CROWNS_FIN,
                algorithm="PAEK",
                tolerance=PAEK_TOL
            )
            exists_or_fail(CROWNS_FIN, "CROWNS_FIN")
            out_fc = CROWNS_FIN
        else:
            out_fc = CROWNS_RAW

        # 8) Stats
        n = int(arcpy.management.GetCount(out_fc)[0])
        if n == 0:
            status = "FAIL_EMPTY"
            areas, diams = [], []
        else:
            areas, diams = collect_area_and_diam(out_fc, sample_max_features=SAMPLE_MAX_FEATURES)

        area_stats = summarize(areas)
        diam_stats = summarize(diams)
        total_area = float(sum(areas)) if areas else 0.0
        dt = time.time() - t0

        row = {
            "CELL": CELL, "HMIN": HMIN, "SMOOTH": SMOOTH, "FMAX": FMAX, "EPS": EPS,
            "status": status, "seconds": dt,
            "seed_ratio": seed_ratio, "errmsg": "",
            "n": int(diam_stats["n"]),
            "total_area_m2": total_area,
            "diam": diam_stats,
            "area": area_stats,
            "out_fc": out_fc
        }

        # Persist to results table
        log_result(
            RESULTS_TABLE,
            (CELL, HMIN, SMOOTH, FMAX, EPS,
             float(seed_ratio),
             int(diam_stats["n"]),
             float(diam_stats["max"]),
             float(diam_stats["mean"]),
             float(diam_stats["p95"]),
             float(total_area),
             float(dt),
             out_fc,
             status,
             "")
        )

        # Cleanup intermediates unless you want to keep everything
        if not KEEP_ALL_OUTPUTS:
            for p in [SEEDS_RAS, WS_RAS, WS_CLEAN]:
                if arcpy.Exists(p):
                    arcpy.management.Delete(p)
            if arcpy.Exists(CROWNS_FIN):
                arcpy.management.Delete(CROWNS_FIN)

        return row

    except Exception as e:
        dt = time.time() - t0
        if not status.startswith("SKIP"):
            status = "ERROR"
        errmsg = (str(e) or "unknown error")[:255]

        # Log failure
        try:
            log_result(
                RESULTS_TABLE,
                (CELL, HMIN, SMOOTH, FMAX, EPS,
                 float(seed_ratio),
                 0, 0.0, 0.0, 0.0, 0.0,
                 float(dt),
                 "",
                 status,
                 errmsg)
            )
        except:
            pass

        # Cleanup this combo
        for p in [SEEDS_RAS, WS_RAS, WS_CLEAN, CROWNS_RAW, CROWNS_FIN]:
            if arcpy.Exists(p):
                arcpy.management.Delete(p)

        return {
            "CELL": CELL, "HMIN": HMIN, "SMOOTH": SMOOTH, "FMAX": FMAX, "EPS": EPS,
            "status": status, "seconds": dt,
            "seed_ratio": seed_ratio, "errmsg": errmsg,
            "n": 0, "total_area_m2": 0.0,
            "diam": summarize([]), "area": summarize([]),
            "out_fc": ""
        }

# =============================================================================
# MAIN
# =============================================================================
run_rows = []
best_row = None
pass_row = None

try:
    # Stage 1 combos
    stage1_combos = [(c, h, s, f, e)
                     for c in CELL_STAGE1
                     for h in HMIN_STAGE1
                     for s in SMOOTH_STAGE1
                     for f in FMAX_STAGE1
                     for e in EPS_STAGE1]
    stamp(f"Stage 1 combos: {len(stage1_combos):,}")

    for (CELL, HMIN, SMOOTH, FMAX, EPS) in tqdm(stage1_combos, desc="Stage 1 (coarse)", unit="combo"):
        row = run_one_combo(CELL, HMIN, SMOOTH, FMAX, EPS, stage_tag="S1")
        run_rows.append(row)

        # First PASS encountered (informational)
        if pass_row is None and row["status"] == "OK" and row["n"] > 0 and row["diam"]["max"] <= MAX_UK_CROWN_DIAM_M:
            pass_row = row
            stamp("✔ PASS achieved in Stage 1 (first PASS row).")

        # Update BEST using PASS-first/high-n rule (not smallest max)
        if is_better(row, best_row, MAX_UK_CROWN_DIAM_M):
            best_row = row

    # Stage 2 (refine) around best row
    if (not COARSE_ONLY) and (best_row is not None):
        bCELL, bHMIN, bSMOOTH, bFMAX, bEPS = best_row["CELL"], best_row["HMIN"], best_row["SMOOTH"], best_row["FMAX"], best_row["EPS"]

        # Refine EPS around best EPS
        eps_ref = []
        for off in EPS_OFFSETS:
            v = round(float(bEPS) + float(off), 2)
            if 0.05 <= v <= 1.50:
                eps_ref.append(v)
        eps_ref = sorted(set(eps_ref))

        # Put the best values first, then other candidates
        smooth_ref = [bSMOOTH] + [x for x in SMOOTH_REFINE if x != bSMOOTH]
        fmax_ref   = [bFMAX] + [x for x in FMAX_REFINE if x != bFMAX]
        hmin_ref   = [bHMIN] + [x for x in HMIN_REFINE if x != bHMIN]
        cell_ref   = [bCELL] + [x for x in CELL_REFINE if x != bCELL]

        stage2_combos = [(c, h, s, f, e)
                         for c in cell_ref
                         for h in hmin_ref
                         for s in smooth_ref
                         for f in fmax_ref
                         for e in eps_ref]
        stamp(f"Stage 2 combos: {len(stage2_combos):,}")

        for (CELL, HMIN, SMOOTH, FMAX, EPS) in tqdm(stage2_combos, desc="Stage 2 (refine)", unit="combo"):
            row = run_one_combo(CELL, HMIN, SMOOTH, FMAX, EPS, stage_tag="S2")
            run_rows.append(row)

            if pass_row is None and row["status"] == "OK" and row["n"] > 0 and row["diam"]["max"] <= MAX_UK_CROWN_DIAM_M:
                pass_row = row
                stamp("✔ PASS achieved in Stage 2 (first PASS row).")

            if is_better(row, best_row, MAX_UK_CROWN_DIAM_M):
                best_row = row

except Exception:
    stamp("FATAL ERROR (full traceback):")
    print(traceback.format_exc())

finally:
    meta = {
        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "GDB": GDB,
        "CHM": CHM,
        "AOI": AOI,
        "MAX_UK_CROWN_DIAM_M": MAX_UK_CROWN_DIAM_M,
        "SEED_RATIO_MIN": SEED_RATIO_MIN,
        "SEED_RATIO_MAX": SEED_RATIO_MAX,
        "GATE_RES_M": GATE_RES_M,
        "SAMPLE_MAX_FEATURES": SAMPLE_MAX_FEATURES,
        "DO_POLY_SMOOTH_SWEEP": DO_POLY_SMOOTH_SWEEP,
        "DO_POLY_SMOOTH_FINAL": DO_POLY_SMOOTH_FINAL,
        "PAEK_TOL": PAEK_TOL,
        "KEEP_ALL_OUTPUTS": KEEP_ALL_OUTPUTS,
        "COARSE_ONLY": COARSE_ONLY,
        "RECOMMENDED_DEFAULTS": str(RECOMMENDED),
        "RESULTS_TABLE": RESULTS_TABLE
    }

    stamp(f"Writing HTML report: {REPORT_HTML}")
    write_html_report(REPORT_HTML, meta, run_rows, best_row=best_row, pass_row=pass_row)
    stamp(f"✔ Report written: {REPORT_HTML}")
    stamp(f"Results table: {RESULTS_TABLE}")

    # Smooth once on the final chosen output (optional)
    if DO_POLY_SMOOTH_FINAL and best_row and best_row.get("out_fc"):
        out_fc = best_row["out_fc"]
        smooth_fc = out_fc + "_SMOOTH"
        if arcpy.Exists(smooth_fc):
            arcpy.management.Delete(smooth_fc)

        stamp(f"Smoothing BEST crowns once (PAEK {PAEK_TOL})...")
        arcpy.cartography.SmoothPolygon(out_fc, smooth_fc, "PAEK", PAEK_TOL)
        stamp(f"Final smoothed output: {smooth_fc}")

    # Console summary
    if best_row:
        stamp("BEST run (PASS-first, then highest n, then smallest p95):")
        stamp(f"  Params: CELL={best_row['CELL']}, HMIN={best_row['HMIN']}, SMOOTH={best_row['SMOOTH']}, FMAX={best_row['FMAX']}, EPS={best_row['EPS']}")
        stamp(f"  Status: {best_row['status']}")
        stamp(f"  Seed ratio (approx): {best_row.get('seed_ratio',0):.6f}")
        stamp(f"  n: {best_row['n']:,}")
        stamp(f"  Max diam: {best_row['diam']['max']:.2f} m | P95: {best_row['diam']['p95']:.2f} m | Mean: {best_row['diam']['mean']:.2f} m")
        stamp(f"  Output FC: {best_row['out_fc']}")
    else:
        stamp("No successful crowns were produced. Check ERRMSG in the HTML/table for the first failing tool call.")

    try:
        import winsound
        winsound.Beep(880, 500)
    except:
        pass


[17:49:14] AOI enabled: C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass.gdb\AOI_poly
[17:49:15] CHM range: min=-17.8419952392578, max=30.4289970397949
[17:49:19] Stage 1 combos: 54


Stage 1 (coarse):   0%|          | 0/54 [00:00<?, ?combo/s]

[17:49:35] ✔ PASS achieved in Stage 1 (first PASS row).


Stage 1 (coarse): 100%|██████████| 54/54 [18:45<00:00, 20.83s/combo]﻿


[18:08:04] Stage 2 combos: 2,160


Stage 2 (refine):  50%|█████     | 1080/2160 [5:44:49<4:13:37, 14.09s/combo]

[23:52:54] Resampling CHM to 2 m...


Stage 2 (refine): 100%|██████████| 2160/2160 [12:01:39<00:00, 20.05s/combo]﻿ 


[06:09:43] Writing HTML report: C:\ArcProj\AboveGroundBiomass\reports\crown_sweep_fast_20260216_174914.html
[06:09:43] ✔ Report written: C:\ArcProj\AboveGroundBiomass\reports\crown_sweep_fast_20260216_174914.html
[06:09:43] Results table: C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass.gdb\crown_sweep_20260216_174914
[06:09:43] Smoothing BEST crowns once (PAEK 3 Meters)...
[06:09:47] Final smoothed output: C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass.gdb\CROWNS_S2_c1_h3_0_s0_f4_e0_9_20260216_174914_SMOOTH
[06:09:47] BEST run (PASS-first, then highest n, then smallest p95):
[06:09:47]   Params: CELL=1, HMIN=3.0, SMOOTH=0, FMAX=4, EPS=0.9
[06:09:47]   Status: OK
[06:09:47]   Seed ratio (approx): 0.017303
[06:09:47]   n: 2,600
[06:09:47]   Max diam: 27.32 m | P95: 7.59 m | Mean: 2.75 m
[06:09:47]   Output FC: C:\ArcProj\AboveGroundBiomass\AboveGroundBiomass.gdb\CROWNS_S2_c1_h3_0_s0_f4_e0_9_20260216_174914
